# 原 Stage1 组成账户对 SOXX 的逐段收益归因

冻结来源及预算见 `plan.json`。全路径只读 Qlib 原生账户，先要求每日收益逐段闭合，再描述现金机会成本、SOXL 相对选择、直接费用及显式残差；不生成可交易反事实。

In [1]:
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources
from src.analysis.etf_transfer_attribution import holdings_attribution
from src.data.minute_intake import atomic_json
root=Path.cwd();card=root/'research/stage1_beta_composition_account_contributions_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
source=root/plan['source_card'];helper=root/plan['helper_card'];ids=plan['source_identities']
for key,file in [('source_summary_sha256','summary.json'),('source_manifest_sha256','verification_manifest.json'),('source_readback_sha256','readback_verification.json')]:assert sha(source/file)==ids[key]
assert sha(helper/'account_source_preflight.json')==ids['account_helper_receipt_sha256']
assert sha(root/'src/analysis/etf_transfer_attribution.py')==ids['attribution_module_sha256']
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==ids['provider_manifest_sha256']
summary=json.loads((source/'summary.json').read_text());assert summary['decision']=='not_promoted_fixed_Stage1_beta_composition_0bp'
assert summary['recorder_id']==plan['source_recorder_id'] and summary['experiment_id']==plan['source_experiment_id']
art=source/'qlib_recorder'/summary['experiment_id']/summary['recorder_id']/'artifacts'
assert sha(art/'native/candidate_0')==ids['native_candidate_artifact_sha256']
assert sha(art/'paths/candidate_0')==ids['path_candidate_artifact_sha256']
account=prepare_account_sources(root=root,card=helper)
assert sha(helper/'account_source_preflight.json')==ids['account_helper_receipt_sha256']
with R.uri_context(str(source/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=plan['source_recorder_id'],experiment_id=plan['source_experiment_id'])
 book=rec.load_object('native/candidate_0');path=rec.load_object('paths/candidate_0')
assert book['audit']['status']=='passed' and len(book['authority'])==len(path['replay'])==944
assert len(account['points'])==3778 and len(book['points'])==3778
for role in ('SOXX','Stage1_A'):
 pd.testing.assert_frame_equal(account['references'][role]['authority'][['signal_date','trade_date','exit_date']].reset_index(drop=True),
                               book['authority'][['signal_date','trade_date','exit_date']].reset_index(drop=True))
print({'native_sessions':944,'price_marks':len(account['points']),'source_recorder_id':summary['recorder_id']})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[42727:MainThread](2026-10-23 01:04:17,932) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[42727:MainThread](2026-10-23 01:04:17,934) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[42727:MainThread](2026-10-23 01:04:17,934) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/core_1min')}


{'native_sessions': 944, 'price_marks': 3778, 'source_recorder_id': '4cd2029dc98740349e34db8c7c6eae3e'}


In [2]:
raw_targets=path['targets'].set_index('signal_date')
daily,segments=holdings_attribution(book,account['points'],account['schedule'],raw_targets,initial_cash=plan['initial_cash_usd'] if 'initial_cash_usd' in plan else 100000)
assert len(daily)==944 and len(segments)==944*2*4
assert daily.closure_error.abs().max()<1e-9
ref=account['references']['SOXX']['authority'].reset_index(drop=True)
assert pd.DatetimeIndex(ref.trade_date).equals(pd.DatetimeIndex(daily.trade_date))
benchmark=ref.strategy_return.to_numpy(float)
previous=np.r_[100000.,book['authority'].equity.to_numpy(float)[:-1]]
points=book['points'].set_index('stamp');prices=account['points']
post=pd.DatetimeIndex(daily.trade_date)+pd.Timedelta(hours=9,minutes=47)
assert post.isin(points.index).all() and post.isin(prices.index).all()
mark=points.loc[post]
soxl_prices=prices.loc[post,'SOXL'].to_numpy(float)
w_soxl=mark.q_soxl.to_numpy(float)*soxl_prices/previous
w_cash=(mark.cash.to_numpy(float)+mark.receivable.to_numpy(float))/previous
assert np.isfinite(w_soxl).all() and np.isfinite(w_cash).all()
candidate=daily['return'].to_numpy(float)
assert np.all(candidate>-1) and np.all(benchmark>-1)
cash=-w_cash*benchmark
selection=daily.SOXL_contribution.to_numpy(float)-w_soxl*benchmark
fees=daily.fee_contribution.to_numpy(float)
relative=candidate-benchmark
residual=relative-cash-selection-fees
scale=np.divide(np.log1p(candidate)-np.log1p(benchmark),relative,out=1/(1+benchmark),where=np.abs(relative)>1e-12)
paired=np.log1p(candidate)-np.log1p(benchmark)
components=pd.DataFrame({'signal_date':book['authority'].signal_date,'trade_date':daily.trade_date,'exit_date':daily.exit_date,
 'candidate_return':candidate,'SOXX_return':benchmark,'paired_log_excess':paired,
 'post_open_cash_weight':w_cash,'post_open_soxl_weight':w_soxl,'cash_opportunity_log':cash*scale,
 'SOXL_selection_log':selection*scale,'candidate_direct_fees_log':fees*scale,'timing_and_benchmark_residual_log':residual*scale,
 'risk_state':daily.risk_state})
cols=['cash_opportunity_log','SOXL_selection_log','candidate_direct_fees_log','timing_and_benchmark_residual_log']
assert np.max(np.abs(components[cols].sum(axis=1)-components.paired_log_excess))<1e-12
assert abs(components.paired_log_excess.sum()-summary['SOXX_metrics']['log_excess_sum'])<1e-9
print({'max_native_daily_return_closure_error':float(daily.closure_error.abs().max()),'full_SOXX_relative_log':float(components.paired_log_excess.sum()),'log_components':{k:float(components[k].sum()) for k in cols}})


{'max_native_daily_return_closure_error': 4.2327252813834093e-16, 'full_SOXX_relative_log': 0.07460143774129058, 'log_components': {'cash_opportunity_log': -9463.690204616243, 'SOXL_selection_log': -10342.886301830813, 'candidate_direct_fees_log': -0.006696129287773061, 'timing_and_benchmark_residual_log': 19806.65780401409}}


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_42727/1534704771.py:10: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  post=pd.DatetimeIndex(daily.trade_date)+pd.Timedelta(hours=9,minutes=47)


In [3]:
def describe(group,kind,period,partial=False):
 return {'kind':kind,'period':str(period),'partial_boundary_period':bool(partial),'sessions':len(group),
  'first_trade':str(pd.Timestamp(group.trade_date.iloc[0]).date()),'last_exit':str(pd.Timestamp(group.exit_date.iloc[-1]).date()),
  'relative_log_excess':float(group.paired_log_excess.sum()),'relative_wealth':float(np.exp(group.paired_log_excess.sum())),
  **{k:float(group[k].sum()) for k in cols},
  'mean_post_open_cash_weight':float(group.post_open_cash_weight.mean()),'mean_post_open_soxl_weight':float(group.post_open_soxl_weight.mean()),
  'risk_state_ge2_fraction':float(group.risk_state.ge(2).mean())}
period_rows=[describe(components,'all','all')]
for freq,kind in [('Y','year'),('Q','quarter')]:
 groups=components.groupby(components.trade_date.dt.to_period(freq),sort=True)
 for period,g in groups:
  is_partial=(period==components.trade_date.iloc[0].to_period(freq) or period==components.trade_date.iloc[-1].to_period(freq))
  period_rows.append(describe(g,kind,period,is_partial))
windows=[describe(components.iloc[i:i+126],'rolling126',components.trade_date.iloc[i]) for i in range(len(components)-125)]
assert len(windows)==819 and sum(x['kind']=='quarter' for x in period_rows)==16
periods=pd.DataFrame(period_rows);rolling=pd.DataFrame(windows)
assert periods.loc[periods.kind.eq('all'),'sessions'].iloc[0]==944
assert np.max(np.abs(periods[cols].sum(axis=1)-periods.relative_log_excess))<1e-10
assert np.max(np.abs(rolling[cols].sum(axis=1)-rolling.relative_log_excess))<1e-10
full=periods.iloc[0]
residual_abs_ratio=float(components.timing_and_benchmark_residual_log.abs().sum()/components.paired_log_excess.abs().sum())
report={'status':'native_account_component_closure_descriptive_only','source_recorder_id':summary['recorder_id'],'plan_sha256':sha(card/'plan.json'),
 'daily_rows':944,'quarters':16,'rolling126_windows':819,'full_relative_log_excess':float(full.relative_log_excess),
 'full_relative_wealth':float(full.relative_wealth),'full_log_components':{k:float(full[k]) for k in cols},
 'max_native_daily_return_closure_error':float(daily.closure_error.abs().max()),
 'max_daily_log_component_closure_error':float(np.max(np.abs(components[cols].sum(axis=1)-components.paired_log_excess))),
 'residual_absolute_magnitude_relative_to_paired_log':residual_abs_ratio,
 'negative_quarters':periods.loc[(periods.kind=='quarter')&(periods.relative_log_excess<0),'period'].tolist(),
 'worst_126':rolling.loc[rolling.relative_log_excess.idxmin()].to_dict(),
 'no_causal_switch_identified':True,'new_fits':0,'new_accounts':0,'new_bootstraps':0}
atomic_json(report,card/'component_result.json')
periods.to_csv(card/'period_component_summary.csv',index=False)
print({'full':report['full_log_components'],'residual_abs_ratio':residual_abs_ratio,'negative_quarters':report['negative_quarters']})


{'full': {'cash_opportunity_log': -9463.690204616243, 'SOXL_selection_log': -10342.886301830813, 'candidate_direct_fees_log': -0.006696129287773061, 'timing_and_benchmark_residual_log': 19806.65780401409}, 'residual_abs_ratio': 82113.46651118729, 'negative_quarters': ['2023Q2', '2023Q3', '2023Q4', '2024Q2', '2025Q4', '2026Q1', '2026Q3']}


In [4]:
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
import shlex
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder()
 rec.save_objects(artifact_path='inputs',plan=plan,source_identity=ids)
 rec.save_objects(artifact_path='analysis/daily',account=components)
 rec.save_objects(artifact_path='analysis/segments',holdings=segments)
 rec.save_objects(artifact_path='analysis/windows',rolling126=rolling)
 rec.save_objects(artifact_path='analysis',periods=periods,component_result=report)
 saved=rec.load_object('analysis/daily/account')
 assert len(saved)==944 and np.allclose(saved.paired_log_excess,components.paired_log_excess)
 rec.log_metrics(native_sessions=944,rolling_windows=819,new_fits=0,new_accounts=0)
 rid,eid=rec.id,str(rec.experiment_id)
summary_out={'experiment':plan['experiment'],'decision':'descriptive_components_only_no_new_strategy','objective_achieved':False,'result':'component_result.json',
 'source_recorder_id':plan['source_recorder_id'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),
 'recorder_local_dir':str(tracking/eid/rid),'new_fits':0,'new_accounts':0,'new_bootstraps':0,'artifact_policy':plan['artifact_policy']}
binding=json.loads((source/'nas_binding.json').read_text());binding.pop('evidence',None);binding['source_id']=plan['experiment'];binding['source_plan']='plan.json'
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
summary_out['nas_evidence']=evidence;summary_out['nas_readback']={'sha256':remote,'verified':True}
atomic_json(summary_out,card/'summary.json')
print({'recorder_id':rid,'nas_sha256':remote})


[42727:MainThread](2026-10-23 01:04:20,669) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_beta_composition_account_contributions_20261023.


[42727:MainThread](2026-10-23 01:04:20,670) INFO - qlib.workflow - [exp.py:258] - Experiment 103270318570760043 starts running ...


[42727:MainThread](2026-10-23 01:04:20,763) INFO - qlib.workflow - [recorder.py:345] - Recorder 6a1ae1cb25744c8cb2179176088807bd starts running under Experiment 103270318570760043 ...


[42727:MainThread](2026-10-23 01:04:21,885) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'recorder_id': '6a1ae1cb25744c8cb2179176088807bd', 'nas_sha256': 'a6cace950f5aa1a070da8b242ea22f9340d9ccfad10670fed7eaa1b55f57762f'}


## 单位错误与一次机械修复

前面的初次分量输出把美元持仓除以归一化财富，数值无效，不能用于研究结论。原错误和 NAS 哈希保存在 `attempt1_invalid_component_units.json`。以下单元只把分母改为上一期美元 NAV，恢复同一个 Recorder 并重做原生读回；最终有效结果在 `component_result.json`、`readback_verification.json` 与 `conclusion.md`。不新增拟合、账户或策略重放。

In [ ]:
import hashlib,json,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.stage1_nvda_guidance_receiver import prepare_account_sources
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
from src.data.minute_intake import atomic_json
root=Path.cwd();card=root/'research/stage1_beta_composition_account_contributions_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
amend=json.loads((card/'mechanical_repair_1.json').read_text());failure=json.loads((card/'attempt1_invalid_component_units.json').read_text())
assert amend['new_plan_sha256']==sha(card/'plan.json') and amend['old_plan_sha256']==failure['old_plan_sha256']
summary=json.loads((card/'summary.json').read_text());assert summary['recorder_id']==failure['recorder_id'] and summary['nas_evidence']['sha256']==failure['old_nas_archive_sha256']
source=root/plan['source_card'];helper=root/plan['helper_card'];ids=plan['source_identities']
assert sha(helper/'account_source_preflight.json')==ids['account_helper_receipt_sha256']
account=prepare_account_sources(root=root,card=helper)
assert sha(helper/'account_source_preflight.json')==ids['account_helper_receipt_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
assert sha(source/'qlib_recorder'/plan['source_experiment_id']/plan['source_recorder_id']/'artifacts/native/candidate_0')==ids['native_candidate_artifact_sha256']
with R.uri_context(str(source/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=plan['source_recorder_id'],experiment_id=plan['source_experiment_id']);book=rec.load_object('native/candidate_0')
tracking=card/'qlib_recorder';eid=summary['experiment_id'];rid=summary['recorder_id']
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid)
 old=rec.load_object('analysis/daily/account');segments=rec.load_object('analysis/segments/holdings')
assert len(old)==944 and len(segments)==944*8
previous=np.r_[100000.,book['authority'].native_expected_nav.to_numpy(float)[:-1]]
assert previous.min()>90000 and previous.max()<1000000
post=pd.DatetimeIndex(old.trade_date)+pd.Timedelta(hours=9,minutes=47)
points=book['points'].set_index('stamp');prices=account['points']
assert post.isin(points.index).all() and post.isin(prices.index).all()
marks=points.loc[post];w_soxl=marks.q_soxl.to_numpy(float)*prices.loc[post,'SOXL'].to_numpy(float)/previous
w_cash=(marks.cash.to_numpy(float)+marks.receivable.to_numpy(float))/previous
assert np.isfinite(w_soxl).all() and np.isfinite(w_cash).all()
assert w_soxl.min()>=-1e-9 and w_soxl.max()<=1.01 and w_cash.min()>=-1e-9 and w_cash.max()<=1.01
soxl=segments.loc[segments.asset.eq('SOXL')].groupby('trade_date',sort=True).contribution.sum()
soxl=soxl.reindex(pd.DatetimeIndex(old.trade_date)).to_numpy(float)
assert np.isfinite(soxl).all()
fees=-book['orders'].groupby('trade_date').total.sum().reindex(pd.DatetimeIndex(old.trade_date),fill_value=0).to_numpy(float)/previous
candidate=old.candidate_return.to_numpy(float);benchmark=old.SOXX_return.to_numpy(float)
relative=candidate-benchmark
cash=-w_cash*benchmark;selection=soxl-w_soxl*benchmark
residual=relative-cash-selection-fees
scale=np.divide(np.log1p(candidate)-np.log1p(benchmark),relative,out=1/(1+benchmark),where=np.abs(relative)>1e-12)
paired=np.log1p(candidate)-np.log1p(benchmark)
components=old[['signal_date','trade_date','exit_date','candidate_return','SOXX_return','risk_state']].copy()
components['paired_log_excess']=paired;components['post_open_cash_weight']=w_cash;components['post_open_soxl_weight']=w_soxl
components['cash_opportunity_log']=cash*scale;components['SOXL_selection_log']=selection*scale
components['candidate_direct_fees_log']=fees*scale;components['timing_and_benchmark_residual_log']=residual*scale
cols=['cash_opportunity_log','SOXL_selection_log','candidate_direct_fees_log','timing_and_benchmark_residual_log']
assert np.max(np.abs(components[cols].sum(axis=1)-components.paired_log_excess))<1e-12
assert abs(components.paired_log_excess.sum()-json.loads((source/'summary.json').read_text())['SOXX_metrics']['log_excess_sum'])<1e-9

def describe(group,kind,period,partial=False):
 return {'kind':kind,'period':str(period),'partial_boundary_period':bool(partial),'sessions':len(group),
  'first_trade':str(pd.Timestamp(group.trade_date.iloc[0]).date()),'last_exit':str(pd.Timestamp(group.exit_date.iloc[-1]).date()),
  'relative_log_excess':float(group.paired_log_excess.sum()),'relative_wealth':float(np.exp(group.paired_log_excess.sum())),
  **{k:float(group[k].sum()) for k in cols},
  'mean_post_open_cash_weight':float(group.post_open_cash_weight.mean()),'mean_post_open_soxl_weight':float(group.post_open_soxl_weight.mean()),
  'risk_state_ge2_fraction':float(group.risk_state.ge(2).mean())}
period_rows=[describe(components,'all','all')]
for freq,kind in [('Y','year'),('Q','quarter')]:
 for period,g in components.groupby(components.trade_date.dt.to_period(freq),sort=True):
  partial=(period==components.trade_date.iloc[0].to_period(freq) or period==components.trade_date.iloc[-1].to_period(freq))
  period_rows.append(describe(g,kind,period,partial))
windows=[describe(components.iloc[i:i+126],'rolling126',components.trade_date.iloc[i]) for i in range(len(components)-125)]
periods=pd.DataFrame(period_rows);rolling=pd.DataFrame(windows)
assert len(rolling)==819 and sum(periods.kind.eq('quarter'))==16
assert np.max(np.abs(periods[cols].sum(axis=1)-periods.relative_log_excess))<1e-10
assert np.max(np.abs(rolling[cols].sum(axis=1)-rolling.relative_log_excess))<1e-10
full=periods.iloc[0]
residual_abs_ratio=float(components.timing_and_benchmark_residual_log.abs().sum()/components.paired_log_excess.abs().sum())
report={'status':'unit_repaired_native_account_component_closure_descriptive_only','source_recorder_id':plan['source_recorder_id'],'plan_sha256':sha(card/'plan.json'),
 'mechanical_repair_1':'mechanical_repair_1.json','invalid_attempt1':'attempt1_invalid_component_units.json',
 'daily_rows':944,'quarters':16,'rolling126_windows':819,'full_relative_log_excess':float(full.relative_log_excess),
 'full_relative_wealth':float(full.relative_wealth),'full_log_components':{k:float(full[k]) for k in cols},
 'max_native_daily_return_closure_error':json.loads((card/'component_result.json').read_text())['max_native_daily_return_closure_error'],
 'max_daily_log_component_closure_error':float(np.max(np.abs(components[cols].sum(axis=1)-components.paired_log_excess))),
 'residual_absolute_magnitude_relative_to_paired_log':residual_abs_ratio,
 'negative_quarters':periods.loc[(periods.kind=='quarter')&(periods.relative_log_excess<0),'period'].tolist(),
 'worst_126':rolling.loc[rolling.relative_log_excess.idxmin()].to_dict(),
 'no_causal_switch_identified':True,'new_fits':0,'new_accounts':0,'new_bootstraps':0}
assert report['full_relative_wealth']>1 and report['full_relative_wealth']<1.1
assert max(abs(v) for v in report['full_log_components'].values())<10
state_path=card/'repair_run_state.json';assert not state_path.exists(), 'fixed resume already attempted'
atomic_json({'status':'resume_attempted','recorder_id':rid,'experiment_id':eid,'repair_plan_sha256':amend['new_plan_sha256'],'new_accounts':0},state_path)
with R.start(experiment_name=plan['experiment'],uri=str(tracking),recorder_id=rid,resume=True):
 rec=R.get_recorder();assert rec.id==rid and str(rec.experiment_id)==eid
 rec.save_objects(artifact_path='analysis/daily',account=components)
 rec.save_objects(artifact_path='analysis/windows',rolling126=rolling)
 rec.save_objects(artifact_path='analysis',periods=periods,component_result=report)
 rec.log_metrics(component_unit_repair=1,new_fits=0,new_accounts=0)
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid)
 saved=rec.load_object('analysis/daily/account');assert len(saved)==944 and np.allclose(saved.paired_log_excess,paired)
atomic_json(report,card/'component_result.json');periods.to_csv(card/'period_component_summary.csv',index=False)
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'verified':True};summary['mechanical_repair_1']='mechanical_repair_1.json';summary['unit_repaired']=True
atomic_json(summary,card/'summary.json');atomic_json({'status':'completed','recorder_id':rid,'new_nas_archive_sha256':remote,'new_accounts':0},state_path)
print({'full_log_components':report['full_log_components'],'residual_abs_ratio':residual_abs_ratio,'new_nas_archive_sha256':remote})


## 中文结论

只读原生 Qlib 固定组成账户：944 区间、16 季度、819 个连续 126 日窗口，零新拟合/账户/重放。全程相对 SOXX 对数超额 +0.074601；现金机会 -0.094597、SOXL 相对选择 +0.161045、直接费用 -0.006696、显式残差 +0.014850。最差 126 日窗口现金机会 -0.198946，2026Q3 两项主要分量同负。旧单位错误及原 NAS 已保留，修复版 Qlib Recorder `6a1ae1cb25744c8cb2179176088807bd` 和 NAS `068f626ddaee9a0fa99836456252929f85b03de95d43e6f295d49f8c0e69e69c` 读回通过。post-open weight 字段是美元持仓除以上一期美元 NAV 的归因暴露，非当时组合权重；现金及 SOXL 相对项不证明因果或预测技能。残差绝对量占每日配对超额绝对量 1.77%，净残差占全程净超额约 19.9%；精确分量闭合由显式残差构造。描述性路径不识别因果开关；固定 0bp 账户仍不晋升，下一步见 `conclusion.md`。

字段定义和独立复核详见 `interpretation_note.json`。
